In [1]:
# načtení knihoven a vstupních dat z jednotlivých listů Excel souboru

import pandas as pd
import numpy as np
import re

xls = pd.ExcelFile("ozs-zpravy.xlsx")

casti = pd.read_excel("ozs-zpravy.xlsx", sheet_name="casti")
otazky = pd.read_excel("ozs-zpravy.xlsx", sheet_name="otazky")
pobyty = pd.read_excel("ozs-zpravy.xlsx", sheet_name="pobyty")
odpovedi = pd.read_excel("ozs-zpravy.xlsx", sheet_name="odpovedi")

In [2]:
# přejmenování klíčových sloupců pro následné spojování tabulek

casti2 = casti.rename(columns={
    "ID": "CAST_ID_KEY",
    "NAZEV": "CAST_NAZEV",
    "PORADI": "CAST_PORADI"
})

otazky2 = otazky.rename(columns={
    "ID": "OTAZKA_ID_KEY",
    "PORADI": "OTAZKA_PORADI"
})

pobyty2 = pobyty.rename(columns={
    "ID": "POBYT_ID_KEY",
    "OD": "POBYT_OD",
    "DO": "POBYT_DO"
})

In [3]:
# spojení odpovědí s otázkami, částmi zpráv a údaji o pobytu do jednoho datasetu

df = (
    odpovedi
    .merge(
        otazky2[["OTAZKA_ID_KEY", "OTAZKA", "OTAZKA_PORADI", "CAST_ID", "TYP"]],
        left_on="OTAZKA_ID",
        right_on="OTAZKA_ID_KEY",
        how="left"
    )
    .merge(
        casti2[["CAST_ID_KEY", "CAST_NAZEV", "CAST_PORADI"]],
        left_on="CAST_ID",
        right_on="CAST_ID_KEY",
        how="left"
    )
    .merge(
        pobyty2[["POBYT_ID_KEY", "POBYT_OD", "POBYT_DO", "JAZYK", "FAKULTA", "INSTITUCE", "STAT"]],
        left_on="POBYT_ID",
        right_on="POBYT_ID_KEY",
        how="left"
    )
)

df = df[[
    "POBYT_ID",
    "INSTITUCE",
    "STAT",
    "JAZYK",
    "FAKULTA",
    "POBYT_OD",
    "POBYT_DO",
    "CAST_NAZEV",
    "CAST_PORADI",
    "OTAZKA_ID",
    "OTAZKA",
    "OTAZKA_PORADI",
    "TYP",
    "ODPOVED",
    "ZADANO"
]].copy()

df.head()

,POBYT_ID,INSTITUCE,STAT,JAZYK,FAKULTA,POBYT_OD,POBYT_DO,CAST_NAZEV,CAST_PORADI,OTAZKA_ID,OTAZKA,OTAZKA_PORADI,TYP,ODPOVED,ZADANO
0,20468,Örebro University / School of Business,Švédské království (SE),EN,FIS,2022-08-18,2023-01-15,Základní údaje,21,21,Studijní jazyk,21,Řetězec znaků,anglický,2022-03-31 12:54:11
1,20468,Örebro University / School of Business,Švédské království (SE),EN,FIS,2022-08-18,2023-01-15,Ubytování,23,30,Nabízí přijímající škola možnost ubytování na ...,32,Logická hodnota,1,2022-03-31 12:54:11
2,20468,Örebro University / School of Business,Švédské království (SE),EN,FIS,2022-08-18,2023-01-15,Ubytování,23,31,Nabízí přijímající škola nebo s ní spřízněna o...,33,Logická hodnota,0,2022-11-09 18:52:22
3,20468,Örebro University / School of Business,Švédské království (SE),EN,FIS,2022-08-18,2023-01-15,Ubytování,23,135,Jaký typ ubytování jste Vy sám/sama využil/a,30,1 hodnota z výčtu,koleje,2022-04-01 13:04:56
4,20468,Örebro University / School of Business,Švédské království (SE),EN,FIS,2022-08-18,2023-01-15,Doprava,26,48,Jaký způsob dopravy jste použil/a k dopravě na...,48,Více hodnot z výčtu,rychlík či vlak vyšší třídy,2022-03-31 12:54:11


In [4]:
# výběr relevantních sekcí odpovídajícím požadavkům OZS a studentů
relevant_sections = [23, 29, 32, 33, 34]

df = df[df["CAST_PORADI"].isin(relevant_sections)].copy()

In [5]:
# základní vyčištění odpovědí a sjednocení logických hodnot

df = df[
    df["ODPOVED"].notna()
].copy()

df["ODPOVED"] = df["ODPOVED"].astype(str).str.strip()

df = df[
    (df["ODPOVED"] != "") &
    (df["ODPOVED"].str.lower() != "nan")
].copy()


def clean_answer_text(row):
    value = row["ODPOVED"]
    typ = str(row["TYP"]) if pd.notna(row["TYP"]) else ""

    value = str(value).strip()
    value = value.replace("_x000D_", " ")
    value = value.replace("\xa0", " ")
    value = " ".join(value.split())

    # logické odpovědi převedu na ano / ne
    if "Logická hodnota" in typ:
        if value == "1":
            return "ano"
        if value == "0":
            return "ne"

    return value


df["ODPOVED_CLEAN"] = df.apply(clean_answer_text, axis=1)

In [6]:
# odstranění jen skutečně kontaktní závěrečné otázky
final_contact_question_pattern = (
    r"Závěrečná zpráva je anonymní.*zanechat kontakt.*zde je prostor"
)

df = df[
    ~df["OTAZKA"].str.contains(
        final_contact_question_pattern,
        case=False,
        na=False,
        regex=True
    )
].copy()


def mask_phone_if_real(match):
    candidate = match.group(0)
    digits = re.sub(r"\D", "", candidate)

    # okolní text kolem shody
    start = max(0, match.start() - 10)
    end = min(len(match.string), match.end() + 10)
    context = match.string[start:end].lower()

    # pokud je v okolí měna, nejspíš nejde o telefon
    if re.search(r"€|eur|euro|eura|kč|czk|korun|koruny", context):
        return candidate

    # pokud to vypadá jako interval částek, nemaskovat
    if re.fullmatch(r"\d{1,4}\s*[-–—]\s*\d{1,4}", candidate.strip()):
        return candidate

    # maskovat jen skutečně dlouhá čísla
    if len(digits) >= 8:
        return "[odstraněn kontakt]"

    return candidate


def anonymize_free_text(text):
    if pd.isna(text):
        return text

    value = str(text)

    # e-maily
    value = re.sub(
        r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}\b",
        "[odstraněn kontakt]",
        value
    )

    # URL odkazy
    value = re.sub(
        r"https?://\S+|www\.\S+",
        "[odstraněn odkaz]",
        value
    )

    # telefonní čísla
    value = re.sub(
        r"(?<!\w)(?:\+?\d[\d\s()./-]{7,}\d)(?!\w)",
        mask_phone_if_real,
        value
    )

    # instagram / jiné handly
    value = re.sub(
        r"(?<!\w)@[A-Za-z0-9_.]+",
        "[odstraněn kontakt]",
        value
    )

    return " ".join(value.split())


df["ODPOVED_CLEAN"] = df["ODPOVED_CLEAN"].apply(anonymize_free_text)

In [7]:
# výpočet délky pobytu, roku pobytu, průměrné měsíční náklady na pobyt,
# podílu stipendia na celkových nákladech v %,
# obtížnosti zajištění ubytování a měsíční ceny ubytování

def normalize_numeric_token(token):
    if token is None:
        return np.nan

    token = str(token).strip().replace("\xa0", " ")
    token = token.replace(" ", "")

    if token == "":
        return np.nan

    if re.fullmatch(r"\d{1,3}(?:\.\d{3})+,\d+", token):
        token = token.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"\d{1,3}(?:,\d{3})+\.\d+", token):
        token = token.replace(",", "")
    elif re.fullmatch(r"\d{1,3}(?:[.,]\d{3})+", token):
        token = token.replace(".", "").replace(",", "")
    else:
        token = token.replace(",", ".")

    try:
        return float(token)
    except:
        return np.nan


def extract_all_numbers(text):
    if pd.isna(text):
        return []

    value = str(text).replace("\xa0", " ")
    pattern = r"\d+(?:[\s.,]\d{3})*(?:[.,]\d+)?"
    raw_numbers = re.findall(pattern, value)

    numbers = []
    for token in raw_numbers:
        parsed = normalize_numeric_token(token)
        if pd.notna(parsed):
            numbers.append(parsed)

    return numbers


def extract_range_average(text):
    if pd.isna(text):
        return np.nan

    value = str(text).replace("\xa0", " ")

    range_pattern = (
        r"(?P<n1>\d+(?:[\s.,]\d{3})*(?:[.,]\d+)?)\s*"
        r"(?:-|–|—|až|az|to)\s*"
        r"(?P<n2>\d+(?:[\s.,]\d{3})*(?:[.,]\d+)?)"
    )

    match = re.search(range_pattern, value, flags=re.IGNORECASE)

    if not match:
        return np.nan

    n1 = normalize_numeric_token(match.group("n1"))
    n2 = normalize_numeric_token(match.group("n2"))

    if pd.isna(n1) or pd.isna(n2):
        return np.nan

    return (n1 + n2) / 2


def detect_currency(text):
    if pd.isna(text):
        return None

    value = str(text).lower()

    if "kč" in value or "czk" in value or "korun" in value or "koruny" in value:
        return "CZK"

    if "eur" in value or "€" in value or "euro" in value or "eura" in value:
        return "EUR"

    if re.search(r"\d+\s*e\b", value):
        return "EUR"

    return None


def apply_thousand_multiplier_if_needed(text, value):
    if pd.isna(value) or pd.isna(text):
        return value

    t = str(text).lower()

    if re.search(r"\btis\.?\b|\btisíc\b", t):
        return value * 1000

    return value


MONTHLY_COST_QUESTION = "jaké byly vaše průměrné měsíční náklady na pobyt"
SCHOLARSHIP_SHARE_QUESTION = "v jaké výši přispěla přidělená finanční podpora k pokrytí celkových nákladů"
HOUSING_DIFFICULTY_QUESTION = "zajistit ubytování bylo dle vašeho názoru"
HOUSING_PRICE_QUESTION = "uveďte prosím měsíční cenu vašeho druhu ubytování"


def is_amount_question(question):
    return (
        MONTHLY_COST_QUESTION in question or
        HOUSING_PRICE_QUESTION in question
    )


def select_numeric_value_strict(row):
    question = str(row["OTAZKA"]).lower() if pd.notna(row["OTAZKA"]) else ""
    text = row["ODPOVED_CLEAN"]

    if is_amount_question(question):
        currency = detect_currency(text)

        if currency is None:
            return np.nan

        numbers = extract_all_numbers(text)
        range_avg = extract_range_average(text)
        t = str(text).lower()

        if pd.notna(range_avg) and re.search(r"z toho|z nichž|včetně|vcetne", t):
            return apply_thousand_multiplier_if_needed(text, range_avg)

        if pd.notna(range_avg) and len(numbers) > 2:
            return np.nan

        if pd.isna(range_avg) and len(numbers) > 1:
            return np.nan

        if pd.notna(range_avg):
            return apply_thousand_multiplier_if_needed(text, range_avg)

        if len(numbers) > 0:
            return apply_thousand_multiplier_if_needed(text, numbers[0])

        return np.nan

    if SCHOLARSHIP_SHARE_QUESTION in question:
        numbers = extract_all_numbers(text)
        valid = [n for n in numbers if 0 <= n <= 100]
        return valid[0] if len(valid) > 0 else np.nan

    if HOUSING_DIFFICULTY_QUESTION in question:
        numbers = extract_all_numbers(text)
        valid = [n for n in numbers if 1 <= n <= 5]
        return valid[0] if len(valid) > 0 else np.nan

    return np.nan


df["POBYT_OD_DT"] = pd.to_datetime(df["POBYT_OD"], errors="coerce")
df["POBYT_DO_DT"] = pd.to_datetime(df["POBYT_DO"], errors="coerce")
df["DELKA_POBYTU_DNY"] = (df["POBYT_DO_DT"] - df["POBYT_OD_DT"]).dt.days
df["DELKA_POBYTU_MESICE"] = df["DELKA_POBYTU_DNY"] / 30.44

df["ROK_POBYTU"] = df["POBYT_OD_DT"].dt.year

df["ODPOVED_NUM"] = df.apply(select_numeric_value_strict, axis=1)
df["ODPOVED_NUMBERS_ALL"] = df["ODPOVED_CLEAN"].apply(extract_all_numbers)
df["MENA"] = df["ODPOVED_CLEAN"].apply(detect_currency)


exchange_rates = {
    2022: {"CZK": 1.0, "EUR": 24.565},
    2023: {"CZK": 1.0, "EUR": 24.007},
    2024: {"CZK": 1.0, "EUR": 25.119},
    2025: {"CZK": 1.0, "EUR": 24.693}
}


def convert_to_czk(row):
    question = str(row["OTAZKA"]).lower() if pd.notna(row["OTAZKA"]) else ""

    if not is_amount_question(question):
        return np.nan

    value = row["ODPOVED_NUM"]
    currency = row["MENA"]
    year = row["ROK_POBYTU"]

    if pd.isna(value) or currency is None or pd.isna(year):
        return np.nan

    year = int(year)

    if year not in exchange_rates:
        return np.nan

    rate_eur_czk = exchange_rates[year]["EUR"]

    if currency == "CZK":
        return value

    if currency == "EUR":
        return value * rate_eur_czk

    return np.nan


def convert_to_eur(row):
    question = str(row["OTAZKA"]).lower() if pd.notna(row["OTAZKA"]) else ""

    if not is_amount_question(question):
        return np.nan

    value = row["ODPOVED_NUM"]
    currency = row["MENA"]
    year = row["ROK_POBYTU"]

    if pd.isna(value) or currency is None or pd.isna(year):
        return np.nan

    year = int(year)

    if year not in exchange_rates:
        return np.nan

    rate_eur_czk = exchange_rates[year]["EUR"]

    if currency == "EUR":
        return value

    if currency == "CZK":
        return value / rate_eur_czk

    return np.nan


df["ODPOVED_NUM_CZK"] = df.apply(convert_to_czk, axis=1)
df["ODPOVED_NUM_EUR"] = df.apply(convert_to_eur, axis=1)


def build_amount_display(row):
    question = str(row["OTAZKA"]).lower() if pd.notna(row["OTAZKA"]) else ""

    if not is_amount_question(question):
        return row["ODPOVED_CLEAN"]

    value = row["ODPOVED_NUM"]
    currency = row["MENA"]
    value_czk = row["ODPOVED_NUM_CZK"]
    value_eur = row["ODPOVED_NUM_EUR"]

    if pd.isna(value) or currency is None:
        return row["ODPOVED_CLEAN"]

    if currency == "CZK" and pd.notna(value_eur):
        return f"{value:.2f} Kč (CZK), orientačně {value_eur:.2f} EUR"

    if currency == "EUR" and pd.notna(value_czk):
        return f"{value:.2f} EUR, orientačně {value_czk:.2f} Kč (CZK)"

    return row["ODPOVED_CLEAN"]


df["ODPOVED_FINANCE_DISPLAY"] = df.apply(build_amount_display, axis=1)

In [8]:
# souhrnné ukazatele za jednotlivé školy

def extract_rating_1_5(text):
    numbers = extract_all_numbers(text)
    valid = [n for n in numbers if 1 <= n <= 5]
    return valid[0] if len(valid) > 0 else np.nan


school_metrics = []

for (school, country), group in df.groupby(["INSTITUCE", "STAT"]):

    row = {
        "INSTITUCE": school,
        "STAT": country,
        "POCET_ZPRAV": group["POBYT_ID"].nunique(),
        "PRUMER_DELKA_POBYTU_MESICE": group.groupby("POBYT_ID")["DELKA_POBYTU_MESICE"].first().mean()
    }

    monthly_cost = group[
        group["OTAZKA"].str.contains(MONTHLY_COST_QUESTION, case=False, na=False)
    ].copy()

    monthly_cost = monthly_cost[
        monthly_cost["ODPOVED_NUM_CZK"].notna() &
        monthly_cost["ODPOVED_NUM_EUR"].notna()
    ].drop_duplicates(subset=["POBYT_ID"])

    row["POCET_ODPOVEDI_MESICNI_NAKLADY"] = monthly_cost["POBYT_ID"].nunique()
    row["ORIENTACNI_PRUMER_MESICNI_NAKLADY_CZK"] = monthly_cost["ODPOVED_NUM_CZK"].mean()
    row["ORIENTACNI_PRUMER_MESICNI_NAKLADY_EUR"] = monthly_cost["ODPOVED_NUM_EUR"].mean()

    scholarship_share = group[
        group["OTAZKA"].str.contains(SCHOLARSHIP_SHARE_QUESTION, case=False, na=False)
    ].copy()

    scholarship_share = scholarship_share[
        scholarship_share["ODPOVED_NUM"].notna()
    ].drop_duplicates(subset=["POBYT_ID"])

    row["PRUMER_PODIL_STIPENDIA_PROC"] = scholarship_share["ODPOVED_NUM"].mean()

    housing_difficulty = group[
        group["OTAZKA"].str.contains(HOUSING_DIFFICULTY_QUESTION, case=False, na=False)
    ].copy()

    housing_difficulty = housing_difficulty[
        housing_difficulty["ODPOVED_NUM"].notna()
    ].drop_duplicates(subset=["POBYT_ID"])

    row["PRUMERNA_OBTIZNOST_ZAJISTENI_UBYTOVANI"] = housing_difficulty["ODPOVED_NUM"].mean()

    housing_price = group[
        group["OTAZKA"].str.contains(HOUSING_PRICE_QUESTION, case=False, na=False)
    ].copy()

    housing_price = housing_price[
        housing_price["ODPOVED_NUM_CZK"].notna() &
        housing_price["ODPOVED_NUM_EUR"].notna()
    ].drop_duplicates(subset=["POBYT_ID"])

    row["POCET_ODPOVEDI_CENA_UBYTOVANI"] = housing_price["POBYT_ID"].nunique()
    row["ORIENTACNI_PRUMER_CENA_UBYTOVANI_CZK"] = housing_price["ODPOVED_NUM_CZK"].mean()
    row["ORIENTACNI_PRUMER_CENA_UBYTOVANI_EUR"] = housing_price["ODPOVED_NUM_EUR"].mean()

    academic = group[
        group["OTAZKA"].str.contains("jak hodnotíte akademický přínos", case=False, na=False)
    ].copy()
    academic["HODNOCENI"] = academic["ODPOVED_CLEAN"].apply(extract_rating_1_5)
    academic = academic[academic["HODNOCENI"].notna()].drop_duplicates(subset=["POBYT_ID"])
    row["PRUMER_AKADEMICKY_PRINOS"] = academic["HODNOCENI"].mean()

    personal = group[
        group["OTAZKA"].str.contains("jak hodnotíte osobní přínos", case=False, na=False)
    ].copy()
    personal["HODNOCENI"] = personal["ODPOVED_CLEAN"].apply(extract_rating_1_5)
    personal = personal[personal["HODNOCENI"].notna()].drop_duplicates(subset=["POBYT_ID"])
    row["PRUMER_OSOBNI_PRINOS"] = personal["HODNOCENI"].mean()

    overall = group[
        group["OTAZKA"].str.contains("celkové zhodnocení", case=False, na=False)
    ].copy()
    overall["HODNOCENI"] = overall["ODPOVED_CLEAN"].apply(extract_rating_1_5)
    overall = overall[overall["HODNOCENI"].notna()].drop_duplicates(subset=["POBYT_ID"])
    row["PRUMER_CELKOVE_HODNOCENI"] = overall["HODNOCENI"].mean()

    # problémy
    problems = group[
        group["OTAZKA"].str.contains(
            r"setkal/a jste se během studijního pobytu s\s*nějakými",
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    if len(problems) > 0:
        problems = problems.drop_duplicates(subset=["POBYT_ID"])
        row["PODIL_STUDENTU_S_PROBLEMY"] = (
            problems["ODPOVED_CLEAN"].astype(str).str.strip().str.lower().eq("ano").mean() * 100
        )
    else:
        row["PODIL_STUDENTU_S_PROBLEMY"] = np.nan

    dorm = group[
        group["OTAZKA"].str.contains(r"jaký typ ubytování jste.*využil", case=False, na=False)
    ].copy()

    if len(dorm) > 0:
        dorm = dorm.drop_duplicates(subset=["POBYT_ID"])
        row["PODIL_KOLEJ"] = (
            dorm["ODPOVED_CLEAN"].str.strip().str.lower().str.contains("kolej", na=False).mean() * 100
        )
    else:
        row["PODIL_KOLEJ"] = np.nan

    school_metrics.append(row)

metrics_df = pd.DataFrame(school_metrics).sort_values("POCET_ZPRAV", ascending=False)

round_cols = [
    "PRUMER_DELKA_POBYTU_MESICE",
    "ORIENTACNI_PRUMER_MESICNI_NAKLADY_CZK",
    "ORIENTACNI_PRUMER_MESICNI_NAKLADY_EUR",
    "PRUMER_PODIL_STIPENDIA_PROC",
    "PRUMERNA_OBTIZNOST_ZAJISTENI_UBYTOVANI",
    "ORIENTACNI_PRUMER_CENA_UBYTOVANI_CZK",
    "ORIENTACNI_PRUMER_CENA_UBYTOVANI_EUR",
    "PRUMER_AKADEMICKY_PRINOS",
    "PRUMER_OSOBNI_PRINOS",
    "PRUMER_CELKOVE_HODNOCENI",
    "PODIL_STUDENTU_S_PROBLEMY",
    "PODIL_KOLEJ"
]

metrics_df[round_cols] = metrics_df[round_cols].round(2)

metrics_df.head(20)

,INSTITUCE,STAT,POCET_ZPRAV,PRUMER_DELKA_POBYTU_MESICE,POCET_ODPOVEDI_MESICNI_NAKLADY,ORIENTACNI_PRUMER_MESICNI_NAKLADY_CZK,ORIENTACNI_PRUMER_MESICNI_NAKLADY_EUR,PRUMER_PODIL_STIPENDIA_PROC,PRUMERNA_OBTIZNOST_ZAJISTENI_UBYTOVANI,POCET_ODPOVEDI_CENA_UBYTOVANI,ORIENTACNI_PRUMER_CENA_UBYTOVANI_CZK,ORIENTACNI_PRUMER_CENA_UBYTOVANI_EUR,PRUMER_AKADEMICKY_PRINOS,PRUMER_OSOBNI_PRINOS,PRUMER_CELKOVE_HODNOCENI,PODIL_STUDENTU_S_PROBLEMY,PODIL_KOLEJ
85,Management Center Innsbruck (MCI),Rakouská republika (AT),51,4.30,41,24136.76,986.02,59.71,3.76,26,10266.72,420.73,2.47,1.37,1.39,7.84,64.71
232,Wirtschaftsuniversität Wien (Vienna University...,Rakouská republika (AT),37,3.90,27,23851.16,967.42,58.39,2.22,22,14785.97,602.69,1.78,1.49,1.51,2.70,72.97
217,Universität zu Köln (University of Cologne) / ...,Spolková republika Německo (DE),37,4.56,23,25640.20,1041.66,65.21,3.97,16,10817.33,442.31,2.59,1.70,1.84,8.33,35.14
93,Norwegian School of Economics (NHH),Norské království (NO),34,4.90,24,27916.10,1132.12,46.20,1.39,11,9368.89,376.76,1.91,1.50,1.47,14.71,97.06
1,AUDENCIA BUSINESS SCHOOL,Francouzská republika (FR),33,3.58,27,23620.76,961.87,52.52,3.36,15,13091.68,530.00,2.70,1.73,2.03,12.12,6.06
23,EM Strasbourg Business School,Francouzská republika (FR),33,3.69,25,19216.94,778.26,65.07,3.21,16,11199.86,452.31,2.45,1.39,1.61,6.06,81.82
181,University of Michigan / Ross School of Business,Spojené státy americké (US),32,3.86,6,33683.33,1385.37,21.60,2.13,0,NaN,NaN,1.59,1.28,1.47,9.38,0.00
21,EM Lyon Business School,Francouzská republika (FR),31,3.74,21,24955.38,1018.80,49.18,3.79,9,15678.12,633.75,2.40,1.63,1.87,12.90,12.90
43,HAN University of Applied Sciences / Internati...,Nizozemsko (NL),30,4.85,16,19502.88,796.96,68.32,1.47,18,11535.41,468.80,2.17,1.37,1.53,23.33,93.33
98,Queen´s University / Stephen J.R. Smith School...,Kanada (CA),30,3.65,10,28754.50,1191.39,30.14,3.27,2,15000.00,617.30,2.07,1.67,1.67,3.33,0.00


In [9]:
# výběr jedné školy s nejvyšším počtem dostupných zpráv
N_SCHOOLS = 1

selected_school = metrics_df.head(N_SCHOOLS)[["INSTITUCE", "STAT", "POCET_ZPRAV"]].copy()
selected_school

,INSTITUCE,STAT,POCET_ZPRAV
85,Management Center Innsbruck (MCI),Rakouská republika (AT),51


In [10]:
# omezení datasetu pouze na vybranou školu

df_one_school = df.merge(
    selected_school[["INSTITUCE", "STAT"]],
    on=["INSTITUCE", "STAT"],
    how="inner"
).copy()

df_one_school.shape

(2732, 27)

In [11]:
# seskupení textových odpovědí do tematických bloků pro vybranou školu

def collect_grouped_answers(group, pattern, answer_column="ODPOVED_CLEAN"):
    subset = group[group["OTAZKA"].str.contains(pattern, case=False, na=False)].copy()

    if subset.empty:
        return []

    subset = subset.sort_values(["CAST_PORADI", "OTAZKA_PORADI", "POBYT_ID"])
    subset = subset.drop_duplicates(subset=["POBYT_ID", "OTAZKA", answer_column])

    output = []

    for question, qgroup in subset.groupby("OTAZKA", sort=False):
        answers = [
            str(val).strip()
            for val in qgroup[answer_column]
            if str(val).strip() != ""
        ]

        if len(answers) > 0:
            block = question + "\n" + "\n".join([f"- {a}" for a in answers])
            output.append(block)

    return output


one_school_texts = []

for (school, country), group in df_one_school.groupby(["INSTITUCE", "STAT"]):

    row = {
        "INSTITUCE": school,
        "STAT": country,

        "TEXT_UBYTOVANI": collect_grouped_answers(
            group,
            r"kdy jste si zajišťoval/a ubytování|kdo vám osobně se zajištěním ubytování pomáhal|proč jste zvolil/a právě tento druh ubytování|jaké vybavení jste měl/a ve vašem typu ubytování|vaše osobní doporučení.*ubytování"
        ),

        "TEXT_STUDIUM": collect_grouped_answers(
            group,
            r"registrace kurzů|výběru kurzů|setkal/a jste se při výběru kurzů s nějakými potížemi|systém výuky|hodnocení práce studentů|jaké kurzy jste nakonec navštěvoval"
        ),

        "TEXT_ZIVOT_A_CESTOVANI": collect_grouped_answers(
            group,
            r"stravování studentů|nejekonomičtější|ceny základních potravin|ceny jídel|dopravní prostředky|kulturního a sportovního využití|jaké jsou možnosti cestování|jaká místa stojí za to navštívit|stýkal/a jste se více s místními|jak se lze dle vašeho názoru co nejrychleji začlenit|zde prosím uveďte jakékoli další komentáře, doporučení, tipy"
        ),

        "TEXT_NEGATIVNI_ZKUSENOSTI": collect_grouped_answers(
            group,
            r"pokud ano, uveďte typ problému|měl/a jste při těchto problémech koho požádat|zde prosím napište své případné připomínky k fungování programu"
        ),

        "TEXT_FINANCE": collect_grouped_answers(
            group,
            r"pokud máte přehled, prosím rozepište výdaje podle položek|která položka vašeho života v zahraničí byla finančně nejnáročnější|jaké jsou orientační ceny základních potravin|jaké jsou ceny jídel",
            answer_column="ODPOVED_FINANCE_DISPLAY"
        )
    }

    one_school_texts.append(row)

one_school_texts_df = pd.DataFrame(one_school_texts)
one_school_texts_df

,INSTITUCE,STAT,TEXT_UBYTOVANI,TEXT_STUDIUM,TEXT_ZIVOT_A_CESTOVANI,TEXT_NEGATIVNI_ZKUSENOSTI,TEXT_FINANCE
0,Management Center Innsbruck (MCI),Rakouská republika (AT),[Kdy jste si zajišťoval/a ubytování? (před odj...,"[Popište, prosím, způsob registrace kurzů (pře...",[Jaké jsou možnosti stravování studentů (např....,"[Pokud ano, uveďte typ problému (např. zdravot...",[Jaké jsou orientační ceny základních potravin...


In [12]:
# spojení metrik a textových bloků do finální tabulky pro jednu školu

selected_school_key = selected_school[["INSTITUCE", "STAT"]].drop_duplicates()

one_school_data = (
    selected_school_key
    .merge(metrics_df, on=["INSTITUCE", "STAT"], how="left")
    .merge(one_school_texts_df, on=["INSTITUCE", "STAT"], how="left")
)

one_school_data

,INSTITUCE,STAT,POCET_ZPRAV,PRUMER_DELKA_POBYTU_MESICE,POCET_ODPOVEDI_MESICNI_NAKLADY,ORIENTACNI_PRUMER_MESICNI_NAKLADY_CZK,ORIENTACNI_PRUMER_MESICNI_NAKLADY_EUR,PRUMER_PODIL_STIPENDIA_PROC,PRUMERNA_OBTIZNOST_ZAJISTENI_UBYTOVANI,POCET_ODPOVEDI_CENA_UBYTOVANI,...,PRUMER_AKADEMICKY_PRINOS,PRUMER_OSOBNI_PRINOS,PRUMER_CELKOVE_HODNOCENI,PODIL_STUDENTU_S_PROBLEMY,PODIL_KOLEJ,TEXT_UBYTOVANI,TEXT_STUDIUM,TEXT_ZIVOT_A_CESTOVANI,TEXT_NEGATIVNI_ZKUSENOSTI,TEXT_FINANCE
0,Management Center Innsbruck (MCI),Rakouská republika (AT),51,4.3,41,24136.76,986.02,59.71,3.76,26,...,2.47,1.37,1.39,7.84,64.71,[Kdy jste si zajišťoval/a ubytování? (před odj...,"[Popište, prosím, způsob registrace kurzů (pře...",[Jaké jsou možnosti stravování studentů (např....,"[Pokud ano, uveďte typ problému (např. zdravot...",[Jaké jsou orientační ceny základních potravin...


In [13]:
# vytvoření finálního textového vstupu a jeho export do TXT souboru

def filter_redundant_blocks(values):
    if not isinstance(values, list):
        return []

    excluded_patterns = [
        r"jaké byly vaše průměrné měsíční náklady na pobyt",
        r"v jaké výši přispěla přidělená finanční podpora k pokrytí celkových nákladů",
        r"uveďte prosím měsíční cenu vašeho druhu ubytování",
        r"zajistit ubytování bylo dle vašeho názoru",
        r"jak hodnotíte akademický přínos",
        r"jak hodnotíte osobní přínos",
        r"celkové zhodnocení",
        r"setkal/a jste se během studijního pobytu s\s*nějakými",
        r"jaký typ ubytování jste.*využil"
    ]

    filtered = []

    for block in values:
        block_str = str(block).strip()
        if block_str == "":
            continue

        first_line = block_str.split("\n")[0].strip().lower()

        if any(re.search(pattern, first_line, flags=re.IGNORECASE) for pattern in excluded_patterns):
            continue

        filtered.append(block_str)

    return filtered


def safe_list_block(values):
    filtered_values = filter_redundant_blocks(values)

    if len(filtered_values) == 0:
        return "Žádné relevantní odpovědi."

    return "\n\n".join(filtered_values)


final_blocks = []

for _, row in one_school_data.iterrows():

    if pd.notna(row["ORIENTACNI_PRUMER_MESICNI_NAKLADY_CZK"]):
        monthly_cost_text_czk = f"{round(row['ORIENTACNI_PRUMER_MESICNI_NAKLADY_CZK'], 2)} Kč (CZK)"
    else:
        monthly_cost_text_czk = "neuvedeno"

    if pd.notna(row["ORIENTACNI_PRUMER_MESICNI_NAKLADY_EUR"]):
        monthly_cost_text_eur = f"{round(row['ORIENTACNI_PRUMER_MESICNI_NAKLADY_EUR'], 2)} EUR"
    else:
        monthly_cost_text_eur = "neuvedeno"

    if pd.notna(row["ORIENTACNI_PRUMER_CENA_UBYTOVANI_CZK"]):
        housing_price_text_czk = f"{round(row['ORIENTACNI_PRUMER_CENA_UBYTOVANI_CZK'], 2)} Kč (CZK)"
    else:
        housing_price_text_czk = "neuvedeno"

    if pd.notna(row["ORIENTACNI_PRUMER_CENA_UBYTOVANI_EUR"]):
        housing_price_text_eur = f"{round(row['ORIENTACNI_PRUMER_CENA_UBYTOVANI_EUR'], 2)} EUR"
    else:
        housing_price_text_eur = "neuvedeno"

    block = f"""
====================================================================================================
ŠKOLA: {row['INSTITUCE']}
STÁT: {row['STAT']}
POČET ANALYZOVANÝCH ZPRÁV: {int(row['POCET_ZPRAV']) if pd.notna(row['POCET_ZPRAV']) else 'neuvedeno'}

PŘEDPŘIPRAVENÉ ČÍSELNÉ UKAZATELE
- Průměrná délka pobytu v měsících: {round(row['PRUMER_DELKA_POBYTU_MESICE'], 2) if pd.notna(row['PRUMER_DELKA_POBYTU_MESICE']) else 'neuvedeno'}
- Počet odpovědí s uvedenými měsíčními náklady: {int(row['POCET_ODPOVEDI_MESICNI_NAKLADY']) if pd.notna(row['POCET_ODPOVEDI_MESICNI_NAKLADY']) else 'neuvedeno'}
- Orientační průměrné měsíční náklady: {monthly_cost_text_czk}; orientačně {monthly_cost_text_eur}
- Průměrný podíl stipendia na nákladech v %: {round(row['PRUMER_PODIL_STIPENDIA_PROC'], 2) if pd.notna(row['PRUMER_PODIL_STIPENDIA_PROC']) else 'neuvedeno'}
- Průměrná obtížnost zajištění ubytování (1 snadné, 5 obtížné): {round(row['PRUMERNA_OBTIZNOST_ZAJISTENI_UBYTOVANI'], 2) if pd.notna(row['PRUMERNA_OBTIZNOST_ZAJISTENI_UBYTOVANI']) else 'neuvedeno'}
- Počet odpovědí s uvedenou měsíční cenou ubytování: {int(row['POCET_ODPOVEDI_CENA_UBYTOVANI']) if pd.notna(row['POCET_ODPOVEDI_CENA_UBYTOVANI']) else 'neuvedeno'}
- Orientační průměrná měsíční cena ubytování: {housing_price_text_czk}; orientačně {housing_price_text_eur}
- Průměrné hodnocení akademického přínosu (1 velký přínos, 5 žádný nebo malý přínos): {round(row['PRUMER_AKADEMICKY_PRINOS'], 2) if pd.notna(row['PRUMER_AKADEMICKY_PRINOS']) else 'neuvedeno'}
- Průměrné hodnocení osobního přínosu (1 velký přínos, 5 žádný nebo malý přínos): {round(row['PRUMER_OSOBNI_PRINOS'], 2) if pd.notna(row['PRUMER_OSOBNI_PRINOS']) else 'neuvedeno'}
- Průměrné celkové hodnocení pobytu (1 pobyt byl skvělý, 5 pobyt byl špatný): {round(row['PRUMER_CELKOVE_HODNOCENI'], 2) if pd.notna(row['PRUMER_CELKOVE_HODNOCENI']) else 'neuvedeno'}
- Podíl studentů, kteří se během pobytu setkali se závažnými problémy: {round(row['PODIL_STUDENTU_S_PROBLEMY'], 1) if pd.notna(row['PODIL_STUDENTU_S_PROBLEMY']) else 'neuvedeno'} %
- Podíl studentů ubytovaných na koleji: {round(row['PODIL_KOLEJ'], 1) if pd.notna(row['PODIL_KOLEJ']) else 'neuvedeno'} %

TEXTOVÉ PODKLADY – UBYTOVÁNÍ
{safe_list_block(row['TEXT_UBYTOVANI'])}

TEXTOVÉ PODKLADY – PŘEDMĚTY A STUDIJNÍ ZKUŠENOST
{safe_list_block(row['TEXT_STUDIUM'])}

TEXTOVÉ PODKLADY – CESTOVÁNÍ A ŽIVOT V MÍSTĚ
{safe_list_block(row['TEXT_ZIVOT_A_CESTOVANI'])}

TEXTOVÉ PODKLADY – ORIENTAČNÍ FINANČNÍ INFORMACE
{safe_list_block(row['TEXT_FINANCE'])}

TEXTOVÉ PODKLADY – NEGATIVNÍ ZKUŠENOSTI A PROBLÉMY
{safe_list_block(row['TEXT_NEGATIVNI_ZKUSENOSTI'])}
""".strip()

    final_blocks.append(block)

final_text = "\n\n".join(final_blocks)

output_filename = "07_ai_vstup_jedna_skola.txt"

with open(output_filename, "w", encoding="utf-8") as f:
    f.write(final_text)

print(f"Soubor byl uložen jako: {output_filename}")
print(f"Počet škol: {one_school_data.shape[0]}")
print(f"Počet znaků ve vstupu: {len(final_text):,}")

Soubor byl uložen jako: 07_ai_vstup_jedna_skola.txt
Počet škol: 1
Počet znaků ve vstupu: 173,014


In [14]:
with pd.ExcelWriter("07_ai_metriky_jedna_skola.xlsx", engine="openpyxl") as writer:
    metrics_df.to_excel(writer, sheet_name="metriky_vsechny_skoly", index=False)
    selected_school.to_excel(writer, sheet_name="vybrana_skola", index=False)
    one_school_data.to_excel(writer, sheet_name="finalni_data_pro_ai", index=False)

print("Excel s metrikami byl uložen.")

Excel s metrikami byl uložen.


In [15]:
# kontrola měsíčních nákladů

selected_school_name = "Management Center Innsbruck (MCI)"

monthly_debug_one_school = df[
    df["INSTITUCE"].eq(selected_school_name) &
    df["OTAZKA"].str.lower().str.contains(MONTHLY_COST_QUESTION, na=False)
].copy()

monthly_debug_one_school = monthly_debug_one_school[[
    "POBYT_ID",
    "INSTITUCE",
    "ODPOVED_CLEAN",
    "ODPOVED_NUMBERS_ALL",
    "ODPOVED_NUM",
    "MENA",
    "ODPOVED_NUM_CZK",
    "ODPOVED_NUM_EUR"
]].sort_values("POBYT_ID")

used_for_average_one_school = monthly_debug_one_school[
    monthly_debug_one_school["ODPOVED_NUM_CZK"].notna() &
    monthly_debug_one_school["ODPOVED_NUM_EUR"].notna()
].copy()

not_used_for_average_one_school = monthly_debug_one_school[
    monthly_debug_one_school["ODPOVED_NUM_CZK"].isna() |
    monthly_debug_one_school["ODPOVED_NUM_EUR"].isna()
].copy()

print("Použité odpovědi:")
display(
    used_for_average_one_school.sort_values("ODPOVED_NUM_CZK")[
        [
            "POBYT_ID",
            "ODPOVED_CLEAN",
            "ODPOVED_NUMBERS_ALL",
            "ODPOVED_NUM",
            "MENA",
            "ODPOVED_NUM_CZK",
            "ODPOVED_NUM_EUR"
        ]
    ]
)

print("Nepoužité odpovědi:")
display(
    not_used_for_average_one_school[
        [
            "POBYT_ID",
            "ODPOVED_CLEAN",
            "ODPOVED_NUMBERS_ALL",
            "ODPOVED_NUM",
            "MENA",
            "ODPOVED_NUM_CZK",
            "ODPOVED_NUM_EUR"
        ]
    ]
)

print("Počet všech odpovědí k otázce:", len(monthly_debug_one_school))
print("Počet odpovědí, které vstupují do průměru:", len(used_for_average_one_school))
print("Počet odpovědí, které nevstupují do průměru:", len(not_used_for_average_one_school))
print("Průměr v Kč:", round(used_for_average_one_school["ODPOVED_NUM_CZK"].mean(), 2))
print("Průměr v EUR:", round(used_for_average_one_school["ODPOVED_NUM_EUR"].mean(), 2))

Použité odpovědi:


,POBYT_ID,ODPOVED_CLEAN,ODPOVED_NUMBERS_ALL,ODPOVED_NUM,MENA,ODPOVED_NUM_CZK,ODPOVED_NUM_EUR
58060,21455,about 500EUR,[500.0],500.0,EUR,12003.500,500.000000
5107,20587,Řekla bych že průměrně okolo 500 až 600 euro m...,"[500.0, 600.0]",550.0,EUR,13510.750,550.000000
69495,21452,tak 600-700 eur,"[600.0, 700.0]",650.0,EUR,15604.550,650.000000
11420,20589,600-700 EUR,"[600.0, 700.0]",650.0,EUR,15967.250,650.000000
128109,23195,"cca 600-750 Euro, včetně 355 Euro za ubytování","[600.0, 750.0, 355.0]",675.0,EUR,16955.325,675.000000
15095,20590,700 EUR,[700.0],700.0,EUR,17195.500,700.000000
106974,22170,cca 700 - 750 €,"[700.0, 750.0]",725.0,EUR,17405.075,725.000000
88345,21456,cca 700-800 euro,"[700.0, 800.0]",750.0,EUR,18005.250,750.000000
237908,24883,cca 750 EURO,[750.0],750.0,EUR,18519.750,750.000000
118653,23197,řekla bych tak 700 - 800 euro,"[700.0, 800.0]",750.0,EUR,18839.250,750.000000


Nepoužité odpovědi:


,POBYT_ID,ODPOVED_CLEAN,ODPOVED_NUMBERS_ALL,ODPOVED_NUM,MENA,ODPOVED_NUM_CZK,ODPOVED_NUM_EUR
9622,20585,ubyzko 370e a jídlo mohlo být tak 130e bez alk...,"[370.0, 130.0]",NaN,EUR,NaN,NaN
59257,21449,900-1000 vč. nájmu,"[900.0, 1000.0]",NaN,None,NaN,NaN
119723,22167,1000,[1000.0],NaN,None,NaN,NaN
142328,22169,cca 850 € na měsíc (první měsíc více než 1000 €),"[850.0, 1000.0]",NaN,EUR,NaN,NaN
117200,22172,30 - 40tis.,"[30.0, 40.0]",NaN,None,NaN,NaN
143842,22173,1200,[1200.0],NaN,None,NaN,NaN
182095,24042,1200,[1200.0],NaN,None,NaN,NaN
172667,24045,670,[670.0],NaN,None,NaN,NaN
216482,24884,660,[660.0],NaN,None,NaN,NaN


Počet všech odpovědí k otázce: 50
Počet odpovědí, které vstupují do průměru: 41
Počet odpovědí, které nevstupují do průměru: 9
Průměr v Kč: 24136.76
Průměr v EUR: 986.02


In [16]:
# kontrola ukazatele problémů pro jednu školu

selected_school_name = "Management Center Innsbruck (MCI)"

problems_debug = df[
    df["INSTITUCE"].eq(selected_school_name) &
    df["OTAZKA"].str.contains(
        r"setkal/a jste se během studijního pobytu s\s*nějakými",
        case=False,
        na=False,
        regex=True
    )
][[
    "POBYT_ID",
    "INSTITUCE",
    "OTAZKA",
    "ODPOVED_CLEAN"
]].copy()

problems_debug["ODPOVED_NORMALIZED"] = problems_debug["ODPOVED_CLEAN"].astype(str).str.strip().str.lower()
problems_debug["JE_ANO"] = problems_debug["ODPOVED_NORMALIZED"].eq("ano")

display(problems_debug.sort_values("POBYT_ID"))

print("Počet řádků:", len(problems_debug))
print("Počet unikátních pobytů:", problems_debug["POBYT_ID"].nunique())
print("Počet ANO:", problems_debug["JE_ANO"].sum())

if len(problems_debug) > 0:
    print("Podíl studentů s problémy v %:", round(
        problems_debug.drop_duplicates(subset=["POBYT_ID"])["JE_ANO"].mean() * 100, 2
    ))

,POBYT_ID,INSTITUCE,OTAZKA,ODPOVED_CLEAN,ODPOVED_NORMALIZED,JE_ANO
86662,20584,Management Center Innsbruck (MCI),Setkal/a jste se během studijního pobytu s něj...,ne,ne,False
49808,20585,Management Center Innsbruck (MCI),Setkal/a jste se během studijního pobytu s něj...,ano,ano,True
5114,20587,Management Center Innsbruck (MCI),Setkal/a jste se během studijního pobytu s něj...,ne,ne,False
24219,20589,Management Center Innsbruck (MCI),Setkal/a jste se během studijního pobytu s něj...,ano,ano,True
15106,20590,Management Center Innsbruck (MCI),Setkal/a jste se během studijního pobytu s něj...,ne,ne,False
21121,20592,Management Center Innsbruck (MCI),Setkal/a jste se během studijního pobytu s něj...,ne,ne,False
9527,20593,Management Center Innsbruck (MCI),Setkal/a jste se během studijního pobytu s něj...,ne,ne,False
7978,20594,Management Center Innsbruck (MCI),Setkal/a jste se během studijního pobytu s něj...,ne,ne,False
7997,20595,Management Center Innsbruck (MCI),Setkal/a jste se během studijního pobytu s něj...,ne,ne,False
75823,21442,Management Center Innsbruck (MCI),Setkal/a jste se během studijního pobytu s něj...,ne,ne,False


Počet řádků: 51
Počet unikátních pobytů: 51
Počet ANO: 4
Podíl studentů s problémy v %: 7.84


In [17]:
# kontrola metrics_df

display(
    metrics_df.loc[
        metrics_df["INSTITUCE"].eq(selected_school_name),
        ["INSTITUCE", "STAT", "PODIL_STUDENTU_S_PROBLEMY"]
    ]
)

,INSTITUCE,STAT,PODIL_STUDENTU_S_PROBLEMY
85,Management Center Innsbruck (MCI),Rakouská republika (AT),7.84
